# Prediccion OOT

In [1]:
import pandas as pd

dataset_oot_final = pd.read_csv("../data/dataset_oot_final.csv")

C:\Users\jormora\AppData\Local\Temp\ipykernel_25708\294483154.py:3: DtypeWarning: Columns (0: marca_pago_AJUSTES_BANCO, 1: marca_pago_CANCELADO, 2: marca_pago_FACTURACION_MES_SGTE, 3: marca_pago_IGUAL, 4: marca_pago_NO_PAGO, 5: marca_pago_PAGO_MAS, 6: marca_pago_PAGO_MENOS, 7: marca_pago_SIN_FACTURACION, 8: producto_ANTICIPOS, 9: producto_CARTERA MICROCREDITO, 10: producto_CARTERA ORDINARIA, 11: producto_CREDIPAGO, 12: producto_Cartera Consumo, 13: producto_Cartera Microcredito, 14: producto_HIPOTECARIO VIVIENDA, 15: producto_LEASING, 16: producto_LEASING HABITACIONAL, 17: producto_LIBRANZA, 18: producto_LIBRE INVERSION, 19: producto_OTROS HIPOTECARIO, 20: producto_ROTATIVOS, 21: producto_SOBREGIRO, 22: producto_Sobregiro, 23: producto_TARJETA DE CREDITO, 24: producto_TESORERIA, 25: producto_Tarjeta de Crédito, 26: producto_Titularizada, 27: segmento_EMPRESARIAL, 28: segmento_GOBIERNO DE RED, 29: segmento_MICROPYME, 30: segmento_PERSONAL, 31: segmento_PERSONAL PLUS, 32: segmento_PREFER

## Carga del modelo entrenado

Se listan los modelos `.pkl` disponibles en `../models` (cada uno guarda el modelo, las columnas
usadas en entrenamiento y, si aplica, el `scaler`) y se elige uno por indice.

In [2]:
import joblib
from pathlib import Path

# Nombre del archivo .pkl a cargar desde ../models
nombre_modelo = "modelo_extra_trees_normalizado_auc_0.7101.pkl"
ruta_modelo_elegido = Path("../models") / nombre_modelo

modelo_cargado = joblib.load(ruta_modelo_elegido)
modelo = modelo_cargado["modelo"]
columnas_modelo = modelo_cargado["columnas"]
scaler_modelo = modelo_cargado.get("scaler")

print(f"Modelo cargado: {ruta_modelo_elegido.name}")

Modelo cargado: modelo_extra_trees_normalizado_auc_0.7101.pkl


## Preprocesamiento del dataset OOT

Se aplica la misma limpieza de dtypes usada en entrenamiento (columnas one-hot leidas como texto,
infinitos por divisiones tratados como nulos) y se seleccionan unicamente las columnas usadas por
el modelo, en el mismo orden.

In [3]:
import numpy as np

columnas_no_features = ["nit_enmascarado", "num_oblig_enmascarado", "num_oblig_orig_enmascarado"]


def limpiar_dtypes(df):
    df = df.copy()
    for col in df.columns:
        # No numericas/booleanas (incluye object y el dtype "str" que usa pandas al leer el csv)
        es_no_numerica = not (
            pd.api.types.is_numeric_dtype(df[col]) or pd.api.types.is_bool_dtype(df[col])
        )
        if es_no_numerica and col not in columnas_no_features:
            # Normaliza a texto primero: la columna puede mezclar bool nativo y strings "True"/"False"
            df[col] = df[col].astype(str).str.strip().replace({"True": "1", "False": "0"})
            df[col] = pd.to_numeric(df[col], errors="coerce")
    # Algunas variables (ej. porcentajes) quedaron con inf por divisiones; se tratan como nulos
    df = df.replace([np.inf, -np.inf], np.nan)
    return df.fillna(0)


oot_modelo = limpiar_dtypes(dataset_oot_final)
X_oot = oot_modelo[columnas_modelo]

# El scaler (si el modelo lo requiere) se ajusto solo con train; aqui unicamente se transforma
if scaler_modelo is not None:
    X_oot = pd.DataFrame(
        scaler_modelo.transform(X_oot), columns=columnas_modelo, index=X_oot.index
    )

X_oot.shape

(112549, 137)

## Prediccion y generacion del submission

Se predice la probabilidad de la clase positiva (`var_rpta_alt = 1`) y se arma el archivo de
submission con el mismo formato de `sample_submission.csv` (`ID` = `nit_enmascarado#num_oblig_orig_enmascarado#num_oblig_enmascarado`).

In [4]:
proba_oot = modelo.predict_proba(X_oot)[:, 1]
pred_oot = (proba_oot >= 0.5).astype(int)

submission = pd.DataFrame(
    {
        "ID": (
            dataset_oot_final["nit_enmascarado"].astype(str)
            + "#"
            + dataset_oot_final["num_oblig_orig_enmascarado"].astype(str)
            + "#"
            + dataset_oot_final["num_oblig_enmascarado"].astype(str)
        ),
        "var_rpta_alt": pred_oot,
    }
)

ruta_submission = Path("../data/submission_real.csv")
submission.to_csv(ruta_submission, index=False)

print(f"Submission guardado en {ruta_submission.resolve()}")
submission.head()

Submission guardado en C:\Users\jormora\Documents\Documentos\prueba_tecnica\data\submission_real.csv


,ID,var_rpta_alt
0,257335#444821#635511,0
1,59584#350400#730364,1
2,397604#973821#106521,1
3,368086#382995#696856,0
4,255009#434238#645924,0
